In [1]:
!pip install gradio diffusers transformers accelerate torch torchvision gTTS --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.7 MB/s eta 0:00:00


In [2]:
import gradio as gr
import torch
import random
from gtts import gTTS
import tempfile
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from diffusers import StableDiffusionPipeline
from PIL import Image
import random
import os

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [3]:
# Load story generator model
model_id = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

story_pipe  = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


In [5]:
# Load stable diffusion model
diffusion_model_id = "Manojb/stable-diffusion-2-1-base"

sd_pipe = StableDiffusionPipeline.from_pretrained(
    diffusion_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None,
)
if torch.cuda.is_available():
    sd_pipe = sd_pipe.to("cuda")

model_index.json:   0%|          | 0.00/543 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/807 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.46G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

In [6]:
# Helper Functions
global fixed_story
def make_simple_story(topic: str, lines: int = 3):
    prompt = (
        f"Write a simple children's picture book story in {lines} short sentences about {topic}.\n"
        "Make sure each sentence shows a clear action that can be drawn as a picture.\n"
        "Story:\n"
    )
    output = story_pipe(
        prompt,
        max_length=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )[0]["generated_text"]

    story_text = output.split("Story:")[-1].strip()
    story_sentences = story_text.split(".")
    story_sentences = [s.strip() + "." for s in story_sentences if s.strip()]
    return story_sentences[:lines]

def replace_pronouns(sentences, subject):
    pronouns = ["he", "she", "it", "they", "his", "her", "its", "their"]
    replaced_sentences = []
    for sent in sentences:
        words = sent.split()
        new_words = []
        for word in words:
            if word.lower().strip(".,") in pronouns:
                new_words.append(subject)
            else:
                new_words.append(word)
        new_sentence = " ".join(new_words)
        replaced_sentences.append(new_sentence)
    return replaced_sentences

def add_visual_style(sentences):
    styles = [
        "cartoon style",
        "bright colors",
        "cute character design",
    ]
    styled_sentences = []
    for s in sentences:
        style_text = ", ".join(styles)
        styled_prompt = f"{s} {style_text}"
        styled_sentences.append(styled_prompt)
    return styled_sentences

def generate_images(topic, subject):
    basic_story = make_simple_story(topic, lines=3)
    fixed_story = replace_pronouns(basic_story, subject)
    styled_prompts = add_visual_style(fixed_story)

    images = []
    captions = []

    for prompt in styled_prompts:
        print(prompt)
        image = sd_pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
        images.append(image)

        # TTS
        combined_text = " ".join(fixed_story)
        tts = gTTS(combined_text)
        temp_audio = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        tts.save(temp_audio.name)

    return [image for image in images], "\n".join(fixed_story), temp_audio.name

    # return [(img, cap) for img, cap in zip(images, fixed_story)]

In [7]:
# UI
def run_ui(topic, subject):

    images, captions, audio_path = generate_images(topic, subject)
    return images, captions, audio_path

    # outputs = generate_images(topic, subject)
    # return [output[0] for output in outputs], [output[1] for output in outputs]

    # prompt1_text = fixed_story[0]
    # prompt2_text = fixed_story[1]
    # prompt3_text = fixed_story[2]

    # return [output[0] for output in outputs], prompt1_text, prompt2_text, prompt3_text



# with gr.Blocks() as demo:
#     gr.Markdown("# 📚🖼️ Create a Picture Storybook!")
#     gr.Markdown("Enter a **topic** and **subject**. Generate a 3-page picture book automatically!")

#     with gr.Row():
#         topic = gr.Textbox(label="Story Topic (e.g., 'a rabbit who can fly')")
#         subject = gr.Textbox(label="Main Subject (e.g., 'the rabbit')")

#     btn = gr.Button("Generate Storybook 📖✨")

#     gallery = gr.Gallery(label="Generated Pages", columns=3, rows=1, height="auto")
#     captions = gr.Textbox(label="Prompts", interactive=False, lines=3)

#     btn.click(fn=run_ui, inputs=[topic, subject], outputs=[gallery, captions])


In [8]:
#Random function

def generate_random_prompt():
    subjects = [
        "turtle",
        "astronaut",
        "robot",
        "dragon",
        "pumpkin",
        "wizard",
        "penguin",
        "superhero",
        "ghost",
        "doraemon"
    ]

    plots = [
        "goes to space",
        "wants to play",
        "is giving a speech",
        "starts a new school",
        "competes in a dance contest",
        "travels in a time machine",
        "is an artist",
        "has wings",
        "goes to a gingerbread house",
        "joins a pirate crew"
    ]

    subject = random.choice(subjects)
    plot = random.choice(plots)
    return f"{subject} who {plot}"

In [9]:
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # 🎨 TOONIFY: A Text to Comic generator 📚

        **Instructions**:
        - Type a topic and subject in the textboxes below.
        - Click "Generate Comic book 📖✨".
        - View your colorful Comic book images with prompts below! 🌈
        """,
        elem_id="header",
    )

    with gr.Row():
        topic = gr.Textbox(
            label="✨ Enter the Story Topic",
            placeholder="e.g., a rabbit who can fly",
            lines=1,
            interactive=True,
        )
        subject = gr.Textbox(
            label="🌟 Enter the Main Subject",
            placeholder="e.g., the rabbit",
            lines=1,
            interactive=True,
        )

    with gr.Row():
        btn_random = gr.Button("🎲 Generate Random Prompt")

    with gr.Column(visible=False) as prompt_confirm_section:
        random_prompt_box = gr.Textbox(label="Suggested Prompt", interactive=False)
        confirm_text = gr.Markdown("Do you want to use this prompt?")
        with gr.Row():
            btn_use_prompt = gr.Button("✅ Use this Prompt")
            btn_regenerate = gr.Button("🔁 Regenerate")

    btn = gr.Button(
        "Generate your Comic book ✨",
    )

    gallery = gr.Gallery(
        label="The generated Comic",
        columns=3,
        rows=1,
        height="auto",
    )

    captions = gr.Textbox(
        label="Text",
        interactive=False,
        lines=3,
    )

    audio_output = gr.Audio(
        label="Narration",
        type="filepath",
        interactive=False
    )

    def get_random_prompt():
        prompt = generate_random_prompt()
        return gr.update(visible=True), prompt

    def use_prompt(prompt):
        topic_r = prompt
        subject_r = prompt.split()[0]
        images, captions, audio_path = generate_images(topic_r, subject_r)
        return images, captions, audio_path, gr.update(visible=False)

    def regenerate_prompt():
        prompt = generate_random_prompt()
        return gr.update(value=prompt)

    btn.click(fn=run_ui, inputs=[topic, subject], outputs=[gallery, captions, audio_output])
    btn_random.click(fn=get_random_prompt, inputs=[], outputs=[prompt_confirm_section, random_prompt_box])
    btn_regenerate.click(fn=regenerate_prompt, inputs=[], outputs=[random_prompt_box])
    btn_use_prompt.click(fn=use_prompt, inputs=[random_prompt_box], outputs=[gallery, captions, audio_output, prompt_confirm_section])

    # CSS
    demo.css = """
    #header {
        text-align: center;
        font-size: 28px;
        font-family: 'Arial', sans-serif;
        color: #ff6347;
    }

    .gradio-container {
        background-color: #FAE6C7;
        padding: 20px;
    }

    .gradio-row {
        margin-bottom: 20px;
    }

    .gradio-textbox {
        font-size: 16px;
        padding: 10px;
        background-color: beige;
        border: 2px solid #ff6347;
        border-radius: 10px;
    }

    .gradio-textbox input {
        font-size: 16px;
    }

    .gradio-button {
        font-size: 18px;
        padding: 12px;
        margin-top: 20px;
        background-color: #62A8F8;
        color: #62A8F8;
        border-radius: 12px;
        border: none;
    }

    .gradio-button:hover {
        background-color: #215086;
        color: #215086;
    }

    .gradio-gallery .gallery-item {
        border-radius: 12px;
        box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.2);
    }

    .gradio-gallery {
        margin-top: 20px;
    }

    .gradio-textbox {
        font-size: 14px;
        background-color: #faf0e6;
        padding: 15px;
    }
    """


In [10]:
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2757750756c3dffb07.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


An elephant rides a bike. cartoon style, bright colors, cute character design


  0%|          | 0/30 [00:00<?, ?it/s]

The elephant has a big smile on elephant face. cartoon style, bright colors, cute character design


...

One day, rabbit flew over the whole forest and saw a beautiful meadow. cartoon style, bright colors, cute character design


  0%|          | 0/30 [00:00<?, ?it/s]

rabbit decided to land there and have a picnic. cartoon style, bright colors, cute character design


  0%|          | 0/30 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2757750756c3dffb07.gradio.live


In [11]:
demo.close()

Closing server running on port: 7860
